# Sistem Rekomendasi Buku Berbahasa Indonesia Berdasarkan Minat Pembaca
**Metode:** Content-Based Filtering (TF-IDF + Cosine Similarity)  
**Data:** Koleksi buku Gramedia (`gramedia_goodreads_with_rating_ver_3_2.csv`)  
**Mata kuliah:** Kecerdasan Buatan, Politeknik Negeri Sriwijaya

Alur notebook (bisa langsung dipakai untuk Bab IV dan Bab V laporan):

1. Persiapan library
2. Memuat dataset
3. Eksplorasi data (EDA)
4. Pra-pemrosesan data
5. Rekayasa fitur (representasi minat)
6. Membangun model TF-IDF + cosine similarity
7. Fungsi rekomendasi
8. Pengujian dengan 10 data masukan
9. Evaluasi tambahan
10. Menyimpan model untuk Streamlit

**Cara pakai:** jalankan sel dari atas ke bawah (Runtime → Run all). Saat sel nomor 2 dijalankan, unggah file CSV-nya.

## 1. Persiapan Library

In [ ]:
import re
import difflib
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)
print('scikit-learn versi:', sklearn.__version__)  # catat versi ini untuk requirements.txt Streamlit

## 2. Memuat Dataset
Pemisah kolom pada file ini adalah tanda pipa (`|`), **bukan koma**, jadi `read_csv` harus memakai `sep='|'`.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()                 # pilih file gramedia_goodreads_with_rating_ver_3_2.csv
    NAMA_FILE = list(uploaded.keys())[0]
except ImportError:                           # kalau dijalankan di komputer sendiri
    NAMA_FILE = 'gramedia_goodreads_with_rating_ver_3_2.csv'

df_raw = pd.read_csv(NAMA_FILE, sep='|')
print('Jumlah baris, kolom:', df_raw.shape)
df_raw.head()

## 3. Eksplorasi Data (EDA)

In [ ]:
print('Tipe data:')
print(df_raw.dtypes, '\n')
print('Nilai kosong per kolom:')
print(df_raw.isna().sum())

In [ ]:
# Kolom 'kategori' bisa berisi lebih dari satu genre (contoh: 'novel,thriller'), jadi dipecah dulu
genre_hitung = df_raw['kategori'].dropna().str.split(',').explode().value_counts()

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

genre_hitung.head(12).sort_values().plot(kind='barh', ax=ax[0], color='#4C78A8')
ax[0].set_title('12 genre terbanyak')
ax[0].set_xlabel('Jumlah buku')

df_raw['original_rating'].plot(kind='hist', bins=20, ax=ax[1], color='#F58518', edgecolor='white')
ax[1].set_title('Sebaran rating')
ax[1].set_xlabel('Rating')

np.log10(df_raw['rating_count']).plot(kind='hist', bins=20, ax=ax[2], color='#54A24B', edgecolor='white')
ax[2].set_title('Sebaran jumlah pemberi rating (skala log10)')
ax[2].set_xlabel('log10(rating_count)')

plt.tight_layout()
plt.show()

**Catatan dari EDA** (bisa dikutip di laporan):
- Semua buku berbahasa Indonesia (edisi Gramedia), termasuk terjemahan.
- Dataset **tidak memiliki sinopsis**. Representasi "minat" dibangun dari genre, penulis, dan judul.
- Rating hanya berupa rata-rata per buku (bukan rating per pengguna), jadi **collaborative filtering tidak bisa dipakai**. Content-based filtering adalah pilihan yang tepat.
- `rating_count` sangat timpang (ada buku dengan jutaan rating dan ada yang hanya beberapa), jadi perlu penyesuaian saat dipakai sebagai faktor popularitas.

## 4. Pra-pemrosesan Data
Langkah yang dilakukan:
1. Menghapus duplikat (buku yang sama dengan ISBN edisi berbeda). Judul yang sama tetapi **penulis berbeda dianggap buku berbeda**.
2. Mengisi kategori kosong dengan `tidak_diketahui`.
3. Mengubah `jumlah_halaman` menjadi angka.
4. Menghitung *weighted rating* (rata-rata Bayesian) agar buku dengan rating 5.0 dari 3 orang tidak mengalahkan buku dengan rating 4.4 dari ribuan orang.

In [ ]:
def token_penulis(teks):
    """'Dini W.Tamam;Erby S' -> 'diniwtamam erbys' (tiap penulis jadi satu token)."""
    return ' '.join(re.sub(r'[^a-z0-9]', '', p.lower()) for p in str(teks).split(';') if p.strip())

df = df_raw.copy()

# 4.1 Hapus duplikat (judul sama + penulis sama), simpan yang paling banyak dirating
df['kunci'] = df['judul'].str.lower().str.strip() + '|' + df['author'].apply(token_penulis)
sebelum = len(df)
df = (df.sort_values('rating_count', ascending=False)
        .drop_duplicates('kunci')
        .drop(columns='kunci')
        .reset_index(drop=True))
print(f'Duplikat dihapus : {sebelum - len(df)} baris  ({sebelum} -> {len(df)})')

# 4.2 Kategori kosong
df['kategori'] = df['kategori'].fillna('tidak_diketahui').str.lower().str.strip()
print('Kategori kosong  :', (df['kategori'] == 'tidak_diketahui').sum(), 'buku')

# 4.3 Jumlah halaman (ada nilai '-' yang jadi NaN)
df['jumlah_halaman'] = pd.to_numeric(df['jumlah_halaman'], errors='coerce')
df['review_count'] = df['review_count'].fillna(0)

# 4.4 Weighted rating (rata-rata Bayesian)
C = df['original_rating'].mean()               # rata-rata rating seluruh buku
m = df['rating_count'].quantile(0.60)          # ambang minimal jumlah rating
v, R = df['rating_count'], df['original_rating']
df['weighted_rating'] = (v / (v + m)) * R + (m / (v + m)) * C

# Skala 0-1, dipakai sebagai faktor popularitas pada skor akhir
wr = df['weighted_rating'].values
wr_norm = (wr - wr.min()) / (wr.max() - wr.min())

print(f'C = {C:.3f}, m = {m:.0f}')
df[['judul', 'author', 'kategori', 'original_rating', 'rating_count', 'weighted_rating']].head()

## 5. Rekayasa Fitur (Representasi Minat)
Setiap buku diubah menjadi satu teks "profil buku" dari tiga sumber, dengan bobot berbeda:

| Sumber | Bobot (diulang) | Alasan |
| --- | --- | --- |
| Genre/kategori | 3 kali | Penentu utama minat pembaca |
| Penulis | 2 kali | Pembaca sering mengikuti penulis favorit |
| Kata pada judul | 1 kali | Petunjuk tema tambahan |

Nama penulis digabung menjadi satu token (`Tere Liye` → `tereliye`) supaya hanya cocok dengan penulis yang sama persis.

In [ ]:
STOPWORD = {'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'dengan', 'pada', 'ini', 'itu', 'atau', 'oleh',
            'the', 'of', 'and', 'a', 'an', 'in', 'to', 'is', 'or', 'by', 'on', 'at', 'for', 'be', 'with'}

def bersihkan_judul(teks):
    teks = re.sub(r'[^a-z0-9\s]', ' ', str(teks).lower())
    return ' '.join(t for t in teks.split() if t not in STOPWORD)

def buat_fitur(baris):
    kat = '' if baris['kategori'] == 'tidak_diketahui' else baris['kategori'].replace(',', ' ')
    pen = token_penulis(baris['author'])
    return ' '.join([kat] * 3 + [pen] * 2 + [bersihkan_judul(baris['judul'])])

df['fitur'] = df.apply(buat_fitur, axis=1)
df[['judul', 'fitur']].head(5)

## 6. Membangun Model: TF-IDF + Cosine Similarity
- **TF-IDF** mengubah profil teks setiap buku menjadi vektor angka. Kata yang jarang muncul (misalnya nama penulis) mendapat bobot lebih tinggi daripada kata yang umum.
- **Cosine similarity** mengukur kemiripan dua vektor. Nilainya 0 (tidak mirip) sampai 1 (sangat mirip).

In [ ]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['fitur'])

print('Ukuran matriks TF-IDF (buku x kata):', tfidf_matrix.shape)
print('Jumlah kosakata                    :', len(tfidf.vocabulary_))

## 7. Fungsi Rekomendasi
**Skor akhir** = `(1 - alpha) × kemiripan + alpha × weighted_rating_ternormalisasi`

Dengan `alpha = 0.15`, kemiripan minat tetap menjadi faktor utama, sedangkan popularitas dan kualitas (rating) hanya membantu mengurutkan buku yang kemiripannya setara.

Ada dua mode masukan:
- `rekomendasi_dari_minat(...)`: pengguna memilih genre, penulis favorit, atau kata kunci.
- `rekomendasi_dari_judul(...)`: pengguna menyebut satu buku yang disukai.

In [ ]:
KOLOM_TAMPIL = ['judul', 'author', 'kategori', 'original_rating', 'rating_count']

def _urutkan(sim, n, alpha, kecuali=None):
    skor = (1 - alpha) * sim + alpha * wr_norm
    if kecuali is not None:
        skor[kecuali] = -1                      # jangan rekomendasikan buku itu sendiri
    top = np.argsort(skor)[::-1][:n]
    hasil = df.iloc[top][KOLOM_TAMPIL].copy()
    hasil['kemiripan'] = sim[top].round(3)
    hasil['skor_akhir'] = skor[top].round(3)
    hasil.index = range(1, len(hasil) + 1)
    return hasil

def rekomendasi_dari_minat(kategori=None, penulis=None, kata_kunci='', n=10, alpha=0.15):
    """kategori: list genre, contoh ['romance']; penulis: 'Tere Liye'; kata_kunci: 'sherlock detektif'."""
    bagian = []
    if kategori:
        bagian += [' '.join(kategori)] * 3
    if penulis:
        bagian += [token_penulis(penulis)] * 2
    if kata_kunci:
        bagian.append(bersihkan_judul(kata_kunci))
    vektor = tfidf.transform([' '.join(bagian)])
    if vektor.nnz == 0:
        print('Peringatan: tidak ada kata masukan yang dikenali model, hasil hanya berdasarkan rating.')
    sim = cosine_similarity(vektor, tfidf_matrix).ravel()
    return _urutkan(sim, n, alpha)

def cari_indeks_judul(judul):
    kecil = judul.lower().strip()
    cocok = df[df['judul'].str.lower().str.contains(kecil, regex=False)]
    if len(cocok):
        return cocok['rating_count'].idxmax()   # kalau banyak yang cocok, ambil yang paling populer
    mirip = difflib.get_close_matches(kecil, df['judul'].str.lower().tolist(), n=1, cutoff=0.5)
    if mirip:
        return df.index[df['judul'].str.lower() == mirip[0]][0]
    return None

def rekomendasi_dari_judul(judul, n=10, alpha=0.15):
    idx = cari_indeks_judul(judul)
    if idx is None:
        print(f'Judul \"{judul}\" tidak ditemukan.')
        return None
    print('Buku acuan :', df.loc[idx, 'judul'], '|', df.loc[idx, 'author'], '|', df.loc[idx, 'kategori'])
    sim = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).ravel()
    return _urutkan(sim, n, alpha, kecuali=idx)

In [ ]:
print('=== Minat: genre romance ===')
display(rekomendasi_dari_minat(kategori=['romance'], n=5))

print('\n=== Minat: penulis Tere Liye ===')
display(rekomendasi_dari_minat(penulis='Tere Liye', n=5))

print('\n=== Suka buku: Petualangan Sherlock Holmes ===')
display(rekomendasi_dari_judul('Petualangan Sherlock Holmes', n=5))

## 8. Pengujian dengan 10 Data Masukan
Pengujian memakai 10 skenario (7 genre, 1 penulis, 2 judul). Sebuah rekomendasi dianggap **relevan** bila:
- skenario genre → buku tersebut memiliki genre yang diminta,
- skenario penulis → buku tersebut ditulis penulis yang diminta,
- skenario judul → buku tersebut memiliki genre yang sama dengan buku acuan (genre `novel` diabaikan karena terlalu umum) atau ditulis penulis yang sama.

Ukuran yang dipakai adalah **Precision@5** (berapa dari 5 rekomendasi teratas yang relevan). Hasil dianggap **Sesuai** bila minimal 4 dari 5 relevan.

In [ ]:
def genre_spesifik(kat):
    return {t for t in kat.split(',') if t not in ('novel', 'tidak_diketahui')}

def relevan_genre(g):
    return lambda r: g in r['kategori'].split(',')

def relevan_penulis(nama):
    return lambda r: token_penulis(nama) in token_penulis(r['author']).split()

def relevan_judul(judul):
    idx = cari_indeks_judul(judul)
    g_acuan = genre_spesifik(df.loc[idx, 'kategori'])
    p_acuan = set(token_penulis(df.loc[idx, 'author']).split())
    return lambda r: bool(g_acuan & genre_spesifik(r['kategori'])) or bool(p_acuan & set(token_penulis(r['author']).split()))

skenario = [
    ('Minat genre: romance',                 'minat',  dict(kategori=['romance']),          relevan_genre('romance')),
    ('Minat genre: thriller',                'minat',      dict(kategori=['thriller']),         relevan_genre('thriller')),
    ('Minat genre: pengembangan_diri',       'minat',      dict(kategori=['pengembangan_diri']), relevan_genre('pengembangan_diri')),
    ('Minat genre: puisi',                   'minat',      dict(kategori=['puisi']),            relevan_genre('puisi')),
    ('Minat genre: fiksi_ilmiah',            'minat',      dict(kategori=['fiksi_ilmiah']),     relevan_genre('fiksi_ilmiah')),
    ('Minat genre: horor',                   'minat',      dict(kategori=['horor']),            relevan_genre('horor')),
    ('Minat genre: novel_grafis',            'minat',      dict(kategori=['novel_grafis']),     relevan_genre('novel_grafis')),
    ('Penulis favorit: Tere Liye',           'minat',      dict(penulis='Tere Liye'),           relevan_penulis('Tere Liye')),
    ('Buku favorit: Petualangan Sherlock Holmes', 'judul', dict(judul='Petualangan Sherlock Holmes'), relevan_judul('Petualangan Sherlock Holmes')),
    ('Buku favorit: Konspirasi Alam Semesta',     'judul', dict(judul='Konspirasi Alam Semesta'),     relevan_judul('Konspirasi Alam Semesta')),
]

K = 5
baris_uji = []
for no, (nama, jenis, arg, fungsi_relevan) in enumerate(skenario, start=1):
    if jenis == 'judul':
        hasil = rekomendasi_dari_judul(arg['judul'], n=K)
    else:
        hasil = rekomendasi_dari_minat(n=K, **arg)
    relevan = [fungsi_relevan(r) for _, r in hasil.iterrows()]
    p_at_k = sum(relevan) / K
    top3 = ' ; '.join(r['judul'][:35] + ' (' + r['kategori'] + ')' for _, r in hasil.head(3).iterrows())
    baris_uji.append({'No': no, 'Input': nama, 'Top-3 hasil sistem': top3,
                      'Relevan': f'{sum(relevan)}/{K}', 'Precision@5': p_at_k,
                      'Keterangan': 'Sesuai' if p_at_k >= 0.8 else 'Tidak sesuai'})

tabel_uji = pd.DataFrame(baris_uji)
display(tabel_uji)
print('\nRata-rata Precision@%d: %.2f' % (K, tabel_uji['Precision@5'].mean()))
print('Skenario sesuai        : %d dari %d' % ((tabel_uji['Keterangan'] == 'Sesuai').sum(), len(tabel_uji)))

## 9. Evaluasi Tambahan (Tanpa Genre sebagai Fitur)
Pengujian di bagian 8 relatif "mudah" karena genre sendiri menjadi fitur model. Supaya evaluasinya lebih jujur, di sini model dibangun **hanya dari judul dan penulis** (tanpa genre). Lalu diperiksa: dari 10 buku yang dianggap paling mirip, berapa persen yang genre spesifiknya sama dengan buku acuan? Hasilnya dibandingkan dengan **tebakan acak** sebagai pembanding.

In [ ]:
fitur_tanpa_genre = df['judul'].apply(bersihkan_judul) + ' ' + df['author'].apply(token_penulis)
tf_b = TfidfVectorizer()
M_b = tf_b.fit_transform(fitur_tanpa_genre)

rng = np.random.default_rng(42)
kandidat = [i for i in range(len(df)) if genre_spesifik(df.loc[i, 'kategori'])]
sampel = rng.choice(kandidat, size=min(300, len(kandidat)), replace=False)

hit_model, hit_acak = [], []
semua = np.arange(len(df))
for i in sampel:
    g_i = genre_spesifik(df.loc[i, 'kategori'])
    sim = cosine_similarity(M_b[i], M_b).ravel()
    sim = sim + rng.random(len(sim)) * 1e-9          # pecah seri secara acak
    sim[i] = -1
    top = np.argsort(sim)[::-1][:10]
    acak = rng.choice(semua[semua != i], size=10, replace=False)
    hit_model.append(np.mean([bool(g_i & genre_spesifik(df.loc[j, 'kategori'])) for j in top]))
    hit_acak.append(np.mean([bool(g_i & genre_spesifik(df.loc[j, 'kategori'])) for j in acak]))

print(f'Sampel buku diuji                          : {len(sampel)}')
print(f'Rata-rata genre cocok pada 10 teratas (model): {np.mean(hit_model):.2%}')
print(f'Rata-rata genre cocok pada 10 acak (pembanding): {np.mean(hit_acak):.2%}')

## 10. Menyimpan Model untuk Streamlit
File `.joblib` ini yang nanti dimuat oleh aplikasi web. Versi scikit-learn di Streamlit harus **sama** dengan yang tercetak di sel nomor 1.

In [ ]:
paket_model = {
    'df': df,
    'tfidf': tfidf,
    'tfidf_matrix': tfidf_matrix,
    'wr_norm': wr_norm,
    'versi_sklearn': sklearn.__version__,
}
joblib.dump(paket_model, 'model_rekomendasi_buku.joblib', compress=3)
print('Model tersimpan: model_rekomendasi_buku.joblib')

try:
    from google.colab import files
    files.download('model_rekomendasi_buku.joblib')
except ImportError:
    pass

**Langkah berikutnya:** buat antarmuka web dengan Streamlit (`app.py`) yang memuat file `model_rekomendasi_buku.joblib`, lalu tampilkan pilihan genre/penulis/judul dan daftar rekomendasinya.